
# Experiment 2.6 — Product Condition Number Crossover vs Depth

This notebook is the **reporting counterpart** to
`experiments/Tier_1_Core_Mechanism_Tests/PRODUCT_KAPPA_crossover_vs_depth/run_experiment.py`.
It does **not** reimplement the training core. Instead, it imports the script,
runs the canonical `run_experiment()` function, and presents the returned
results in a notebook-friendly form.

## Scope of this first-pass notebook

- **What it studies:** a deterministic, single-seed, fixed-batch toy sweep over
  depths in a deep linear network.
- **Primary metric:**
  \[
  \log \kappa_{\mathrm{prod}} = \sum_{\ell=1}^{L} \log \kappa(W_\ell),
  \]
  i.e. the log of the product of per-layer condition numbers.
- **Comparison:** a historical `'sgd'` baseline key in the script (actually
  deterministic full-batch gradient descent on one fixed synthetic batch) versus
  Muon updates using Newton–Schulz orthogonalized gradients.
- **Crossover definition used here:** the first tracked step from which Muon has
  lower `log kappa_prod` than the baseline on **at least 80% of all remaining
  tracked steps**. This is an **80%-dominance crossover**, not a claim that
  Muon stays better forever thereafter.

## What this notebook does *not* claim

This first pass is **not** a strong test of \(O(n/L^2)\) scaling. The current
study uses one seed, one fixed synthetic problem, and nested depth
initializations rather than independent random draws at each depth. Any
power-law fit shown below is therefore **descriptive of this run**, not a
statistically robust scaling law.



## 1. Notebook-safe setup and script import

The notebook resolves the repository root from the current working directory,
loads the canonical script module by path, and keeps all experiment logic in
that script. This avoids notebook-only copies of the training code and removes
hazards such as dependence on `__file__`.


In [ ]:

from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text


pd.options.display.float_format = '{:.3f}'.format
plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    rel_script = Path('experiments/Tier_1_Core_Mechanism_Tests/PRODUCT_KAPPA_crossover_vs_depth/run_experiment.py')
    for candidate in [start, *start.parents]:
        if (candidate / rel_script).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repository root containing '
        f'{rel_script} starting from {start}'
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'PRODUCT_KAPPA_crossover_vs_depth'
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'

spec = importlib.util.spec_from_file_location('product_kappa_crossover_vs_depth', SCRIPT_PATH)
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)

print(f'Repository root: {REPO_ROOT}')
print(f'Script path: {SCRIPT_PATH}')



## 2. Reproducibility, runtime, and configuration

This cell runs the canonical experiment once and records:

- environment information,
- the exact script entrypoint,
- the script's returned configuration,
- measured runtime, and
- the initial conditioning diagnostics for each tested depth.

The target spectrum in the current code is geometric with singular values
\(\sigma_i = 10 \cdot 0.7^i\). For the present dimensions, that implies an
actual target condition number of about **63,381**, not “about 4,000”.


In [ ]:

run_wall_start = time.perf_counter()
results = experiment.run_experiment(verbose=False)
run_wall_seconds = time.perf_counter() - run_wall_start

config = results['config']
data_summary = results['data_summary']
depths = config['depths']
depth_results = results['depth_results']
fit_summary = results['fit_summary']
summary_df = pd.DataFrame(results['summary_rows']).sort_values('depth').reset_index(drop=True)
initial_df = pd.DataFrame(
    [
        {
            'depth': depth,
            **results['initial_conditioning'][depth],
        }
        for depth in depths
    ]
)

run_info_df = pd.DataFrame(
    [
        ('repository_root', str(REPO_ROOT)),
        ('experiment_directory', str(EXPERIMENT_DIR)),
        ('script_path', str(SCRIPT_PATH)),
        ('reproduction_command', f'python {SCRIPT_PATH.relative_to(REPO_ROOT)}'),
        ('python_version', sys.version.split()[0]),
        ('platform', platform.platform()),
        ('numpy_version', np.__version__),
        ('pandas_version', pd.__version__),
        ('matplotlib_version', plt.matplotlib.__version__),
        ('script_runtime_seconds', results['runtime_seconds']),
        ('notebook_run_cell_wall_seconds', run_wall_seconds),
    ],
    columns=['item', 'value'],
)

config_df = pd.DataFrame(
    [
        {
            'width': config['width'],
            'depths': str(config['depths']),
            'num_steps': config['num_steps'],
            'track_every': config['track_every'],
            'lr_sgd': config['lr_sgd'],
            'lr_muon': config['lr_muon'],
            'ns_iters': config['ns_iters'],
            'batch_size': config['batch_size'],
            'seed': config['seed'],
            'crossover_fraction': config['crossover_fraction'],
            'target_condition_number': data_summary['target_condition_number'],
            'x_shape': str(data_summary['x_shape']),
            'y_shape': str(data_summary['y_shape']),
        }
    ]
)

display(run_info_df)
display(config_df)

display(
    initial_df[
        [
            'depth',
            'min_per_layer_condition_number',
            'max_per_layer_condition_number',
            'log_product_condition_number',
            'product_condition_number',
        ]
    ].rename(
        columns={
            'min_per_layer_condition_number': 'init min layer kappa',
            'max_per_layer_condition_number': 'init max layer kappa',
            'log_product_condition_number': 'init log kappa_prod',
            'product_condition_number': 'init kappa_prod',
        }
    )
)

print('Baseline label retained in returned results:', config['baseline_label'])
print('Baseline meaning:', config['baseline_description'])
print('Crossover definition:', config['crossover_definition'])



### Interpretation of the setup

Three facts matter for reading the figures below.

1. **This is deterministic full-batch training on one fixed synthetic batch.**
   The script retains the historical key name `'sgd'`, but the baseline is not
   minibatch SGD and there is no stochastic gradient noise in the usual sense.
2. **The initial conditioning is already substantial.** The script's own
   diagnostics show that the Xavier-initialized square Gaussian layers in this
   seed are far from \(\kappa \approx 1.6\) in aggregate, and the resulting
   initial product conditioning grows quickly with depth.
3. **Depths are nested by seed.** Reusing the same seed means deeper models
   extend the shallower initial prefixes; the depth sweep is therefore not an
   independent-draw statistical ensemble.



## 3. Per-depth trajectories of \(\log \kappa_{\mathrm{prod}}\)

The central measurement is the step-by-step trajectory of the log product of
layer condition numbers. Lower values indicate milder multiplicative
ill-conditioning across the network.

The dashed vertical line marks the **80%-dominance crossover** when present.
A crossover marker therefore means “Muon is better on at least 80% of remaining
tracked steps”, **not** “Muon remains better at every subsequent step”.


In [ ]:

fig, axes = plt.subplots(len(depths), 1, figsize=(11, 3.0 * len(depths)), sharex=True)
if len(depths) == 1:
    axes = [axes]

for ax, depth in zip(axes, depths):
    row = depth_results[depth]
    steps = np.array(row['trajectory_steps'])
    kappa_sgd = np.array(row['kappa_sgd'])
    kappa_muon = np.array(row['kappa_muon'])

    ax.plot(steps, kappa_sgd, label="baseline GD ('sgd' key)", linewidth=2.0, color='tab:orange')
    ax.plot(steps, kappa_muon, label='Muon', linewidth=2.0, color='tab:blue')

    crossover_step = row['crossover_step']
    if crossover_step is not None:
        crossover_index = row['trajectory_steps'].index(crossover_step)
        ax.axvline(crossover_step, color='tab:green', linestyle='--', linewidth=1.5,
                   label='80%-dominance crossover')
        ax.scatter(
            [crossover_step],
            [row['kappa_muon'][crossover_index]],
            color='tab:green',
            s=45,
            zorder=5,
        )

    ax.set_ylabel('log kappa_prod')
    ax.set_title(
        f"Depth L={depth} | final Δ(sgd-muon)={row['final_difference_sgd_minus_muon']:+.2f} | "
        f"final winner={row['final_win']}"
    )
    ax.grid(True, alpha=0.3)

axes[0].legend(loc='upper left', frameon=True)
axes[-1].set_xlabel('training step')
fig.suptitle('Product-conditioning trajectories by depth', y=1.01, fontsize=14)
fig.tight_layout()
plt.show()



### What the trajectory plots support

- The depth dependence is **not** completely monotone or clean.
- Some depths exhibit an early 80%-dominance crossover for Muon, but that does
  **not** guarantee a lower final log-product-conditioning value.
- The trajectories are therefore compatible with a *suggestive* depth effect,
  but they do **not** by themselves justify a strong scaling-law claim.



## 4. Crossover step versus depth

The script fits the descriptive model
\[
\text{crossover step} = a / L^b
\]
in log-log space using only depths with a valid crossover step greater than
zero. The plot below explicitly annotates the number of fit points
\(n_{\mathrm{valid}}\), because that sample size is central to how seriously the
fit should be taken.


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5.5))

all_depths = depths
valid_depths = fit_summary['valid_depths']
valid_crossovers = fit_summary['valid_crossovers']
missing_depths = [d for d in all_depths if d not in valid_depths]
placeholder_y = config['num_steps'] + 25

if valid_depths:
    ax.scatter(valid_depths, valid_crossovers, color='tab:blue', s=85, label='valid crossover points', zorder=3)

for depth in missing_depths:
    ax.scatter(depth, placeholder_y, marker='x', color='0.5', s=90, zorder=3)
    ax.text(depth, placeholder_y + 12, 'none', color='0.4', ha='center', va='bottom', fontsize=9)

if fit_summary['b'] is not None:
    L_grid = np.linspace(min(valid_depths), max(valid_depths), 200)
    predicted = fit_summary['a'] / (L_grid ** fit_summary['b'])
    ax.plot(L_grid, predicted, color='tab:red', linewidth=2.0, label='descriptive power-law fit')
    fit_text = (
        f"n_valid = {fit_summary['n_valid']} / {len(depths)}\n"
        f"crossover = {fit_summary['a']:.1f} / L^{fit_summary['b']:.2f}\n"
        f"R² = {fit_summary['r_squared']:.4f}"
    )
else:
    fit_text = f"n_valid = {fit_summary['n_valid']} / {len(depths)}\nNo fit available"

ax.text(
    0.03,
    0.97,
    fit_text,
    transform=ax.transAxes,
    va='top',
    ha='left',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
)

ax.set_xlabel('depth L')
ax.set_ylabel('80%-dominance crossover step')
ax.set_title('Crossover-step depth trend (descriptive, single-seed)')
ax.set_ylim(0, config['num_steps'] + 80)
ax.legend(loc='upper right', frameon=True)
ax.grid(True, alpha=0.3)
plt.show()



### How to read the fit

A visually clean line or a high \(R^2\) here is **not sufficient** evidence for
an \(O(n/L^2)\) law, because the fit may be based on very few points and all
points come from one nested-seed sweep. The fit is useful as a **compact
summary of this run**, not as a definitive scaling statement.



## 5. Final-step comparison by depth

The table and plots below separate two related but distinct questions:

1. **Did an 80%-dominance crossover occur?**
2. **Who is better at the final tracked step?**

These need not agree. A depth can show an early crossover yet still end with
Muon worse than the baseline in final `log kappa_prod`.


In [ ]:

summary_view = summary_df[
    [
        'depth',
        'crossover_step',
        'initial_log_product_kappa',
        'final_kappa_sgd',
        'final_kappa_muon',
        'final_difference_sgd_minus_muon',
        'final_win',
        'best_advantage',
        'best_advantage_step',
        'final_loss_sgd',
        'final_loss_muon',
        'final_loss_ratio_muon_over_sgd',
    ]
].rename(
    columns={
        'initial_log_product_kappa': 'init log kappa_prod',
        'final_kappa_sgd': 'final log kappa_prod (sgd)',
        'final_kappa_muon': 'final log kappa_prod (Muon)',
        'final_difference_sgd_minus_muon': 'final Δ log kappa_prod (sgd-muon)',
        'best_advantage': 'peak Δ log kappa_prod (sgd-muon)',
        'best_advantage_step': 'peak-advantage step',
        'final_loss_sgd': 'final loss (sgd)',
        'final_loss_muon': 'final loss (Muon)',
        'final_loss_ratio_muon_over_sgd': 'final loss ratio Muon/sgd',
    }
)

display(summary_view)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

axes[0].plot(summary_df['depth'], summary_df['final_kappa_sgd'], marker='o', linewidth=2.0,
             color='tab:orange', label="baseline GD ('sgd' key)")
axes[0].plot(summary_df['depth'], summary_df['final_kappa_muon'], marker='o', linewidth=2.0,
             color='tab:blue', label='Muon')
axes[0].set_xlabel('depth L')
axes[0].set_ylabel('final log kappa_prod')
axes[0].set_title('Final product-conditioning values')
axes[0].legend(frameon=True)
axes[0].grid(True, alpha=0.3)

final_diff = summary_df['final_difference_sgd_minus_muon']
bar_colors = ['tab:blue' if val > 0 else 'tab:orange' if val < 0 else '0.5' for val in final_diff]
axes[1].bar(summary_df['depth'].astype(str), final_diff, color=bar_colors)
axes[1].axhline(0.0, color='black', linewidth=1.0)
axes[1].set_xlabel('depth L')
axes[1].set_ylabel('final Δ log kappa_prod (sgd - Muon)')
axes[1].set_title('Positive bars mean Muon finishes lower')
axes[1].grid(True, axis='y', alpha=0.3)

fig.tight_layout()
plt.show()



### Interpretation of the final-step summary

The final-step picture is mixed in the current run. That matters because a
strong “Muon wins earlier and stays better” narrative would require both
reliable crossovers **and** a convincingly favorable final-depth summary. In
this first-pass toy study, the evidence is more limited than that.



## 6. Returned hypothesis tests and calibrated decision summary

For continuity with the original script, the script still computes the legacy
five-test battery. The notebook reports those outputs directly, but the final
interpretation is intentionally more cautious than a simple binary PASS/FAIL.


In [ ]:

tests_df = pd.DataFrame(
    [
        {
            'id': test['id'],
            'passed': test['passed'],
            'name': test['name'],
            'details': str(test['details']),
        }
        for test in results['hypothesis_tests']
    ]
)

display(tests_df)

print('Decision label:', results['decision_summary']['label'])
print('Legacy core-pass flag (tests 1 & 2 only):', results['decision_summary']['legacy_core_pass'])
print('Calibrated statement:')
print(results['decision_summary']['statement'])



## 7. Calibrated conclusion


In [ ]:

crossover_count = sum(summary_df['crossover_step'].notna())
final_muon_wins = int((summary_df['final_win'] == 'muon').sum())
num_depths = len(summary_df)

if fit_summary['b'] is None:
    fit_sentence = 'No crossover-depth fit was available.'
else:
    fit_sentence = (
        f"The descriptive fit used n_valid={fit_summary['n_valid']} crossover points and returned "
        f"b={fit_summary['b']:.3f} with R²={fit_summary['r_squared']:.4f}."
    )

conclusion_md = f"""
### Final reading of this notebook

- **Observed in this run:** 80%-dominance crossovers occurred at **{crossover_count}/{num_depths}** tested depths.
- **Final-step outcome:** Muon finished with lower final `log kappa_prod` at **{final_muon_wins}/{num_depths}** depths.
- **Depth-scaling summary:** {fit_sentence}
- **Most defensible claim:** this single-seed deterministic toy sweep is **compatible with** a depth-dependent crossover story, but the evidence is **mixed and underpowered**.
- **What it does *not* establish:** strong support for an \(O(n/L^2)\) law, or a general conclusion that Muon permanently dominates the baseline in product conditioning.

A stronger next step would be to keep the same metrics but run **independent multi-seed depth sweeps** and attach uncertainty bars to crossover statistics and final-depth advantages.
"""

display(Markdown(conclusion_md))
